# Co-folding at Scale: Benchmark, Fine-Tune and Serve on Ray

**AI Drug Discovery & Development Summit 2026 · Hands-on workshop · 90 minutes**

Co-folding models such as Boltz-2 and OpenFold3 predict how a small molecule binds its protein target
from the protein sequence and the molecule's chemical structure alone. Their published accuracy is
measured on curated benchmarks. This workshop asks 2 questions about real drug discovery campaigns:

1. **How well do these models generalize** to a new target and to compounds designed months into a campaign?
2. **Can a small amount of target-specific data close the gap**, and how quickly?

We answer both on **2,147 X-ray crystal structures** from 3 antiviral campaigns, and build the complete
workflow on Ray along the way.

![The workshop end to end](docs/images/workshop.gif)

| § | Topic | Ray library | Min |
|---|---|---|---|
| 0 | Setup | | 3 |
| 1 | Ray in 12 minutes | Ray Core | 12 |
| 2 | The data: 3 drug discovery campaigns | | 5 |
| 3 | Prediction and scoring at scale | Ray Data | 20 |
| 4 | Results: where general models fall short | | 8 |
| 5 | Fine-tuning OpenFold3 on fragment structures | Ray Train | 20 |
| 6 | Serving the fine-tuned model | Ray Serve | 17 |
| 7 | Summary and next steps | | 5 |
| A | Appendix: hyperparameter and inference sweeps | Ray Tune | |

**How to follow along**

- Run cells in order with **Shift+Enter**. Cells marked ⏱ take several minutes while GPU nodes start.
- The presenter runs every GPU step live on a shared L4 cluster. Attendees keep `USE_GPUS = False`:
  the same cells then load stored results and the shared prediction service.
- Scientific background (pose scoring, campaign structure) is in the [README](README.md#background).

## 0. Setup

- The Workspace runs the `aiddd-boltz2-of3` image: Ray, Boltz-2 and 2 OpenFold3 versions (0.5 and 0.4.3).
- This notebook runs on a CPU-only head node. L4 GPU nodes join when a cell requests a GPU and leave when idle.
- Data, model weights and all benchmark results live on shared storage.

In [ ]:
import os, sys, json, time, warnings, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import ray

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)

from aiddd_pipeline import (SHARED, TARGETS, BOLTZ_CACHE, OF3_OB_CKPT, OF3_P2_CKPT, OF3_FT_CKPT, OF3_LEGACY,
                            Boltz2Predictor, OF3Predictor, load_ligands, validate, make_scorer)
import workshop_helpers as wh

# Data and weights are downloaded once, ahead of the workshop (README step 2)
needed = [TARGETS[t].path("metadata") for t in wh.TARGET_KEYS] + [BOLTZ_CACHE, OF3_OB_CKPT, OF3_P2_CKPT, OF3_FT_CKPT]
missing = [str(p) for p in needed if not p.exists()]
assert not missing, f"Missing {missing}. Run `python scripts/setup_data.py` and `python scripts/prepare_mpro.py` first."

# True: run every GPU step live (the presenter, or anyone with their own GPUs).
# False: GPU steps load stored results, and Section 6 uses the shared service.
USE_GPUS = False
SERVICE_URL = ""      # the shared prediction service for Section 6; the presenter shares this address
GPU = "L4"            # GPU type for every GPU step in this notebook

wh.MINE.mkdir(parents=True, exist_ok=True)
print(f"shared data:  {SHARED}")
print(f"your outputs: {wh.MINE}")

try:
    ray.init(runtime_env={"env_vars": {"AIDDD_SHARED": str(SHARED)}}, ignore_reinit_error=True, logging_level="ERROR")
except ValueError:   # inside an Anyscale Job, which already sets the environment
    ray.init(ignore_reinit_error=True, logging_level="ERROR")
dashboard = os.environ.get("RAY_OVERRIDE_DASHBOARD_URL") or ray.get_runtime_context().dashboard_url
print(f"Ray Dashboard metrics: {dashboard.rstrip('/')}/#/metrics   (open in a new browser tab)")

## 1. Ray in 12 minutes

A single Boltz-2 prediction takes about 1 minute on 1 GPU. The full benchmark (2,147 structures, 4 models,
3 proteins) is several days of work for a single GPU, before any fine-tuning or serving.

Ray runs Python across a cluster. **Ray Core** provides 3 primitives (tasks, actors and objects), and the
Ray AI libraries for data, training, tuning and serving are built on them. This workshop uses all 4 libraries.

<img src="docs/images/ray/map-of-ray.svg" alt="Ray overview" width="460">
<sub>Diagram: Ray documentation, docs.ray.io (Apache 2.0).</sub>

### 1.1 The cluster

- A **head node** runs this notebook (the *driver*) and the **autoscaler**.
- **Worker nodes** join when pending work needs resources and leave when idle.
- Every node runs a scheduler and an **object store**: shared memory that holds results, so other processes
  read them without copying.
- This Workspace has 2 worker types: CPU nodes for cheminformatics and scoring, L4 GPU nodes for inference.

<img src="docs/images/ray/ray-cluster.svg" alt="Ray cluster" width="620">
<sub>Diagram: Ray documentation, docs.ray.io (Apache 2.0).</sub>

In [ ]:
def cluster_table():
    rows = []
    for n in ray.nodes():
        if not n["Alive"]:
            continue
        r = n["Resources"]
        rows.append({"node": n["NodeManagerHostname"], "head": "node:__internal_head__" in r,
                     "CPUs": int(r.get("CPU", 0)), "GPUs": int(r.get("GPU", 0)),
                     "GPU type": next((k.split(":")[1] for k in r if k.startswith("accelerator_type:")), ""),
                     "memory (GB)": round(r.get("memory", 0) / 1e9)})
    return pd.DataFrame(rows).sort_values(["head", "GPUs"], ascending=False).reset_index(drop=True)

cluster_table()   # run again later to watch nodes join and leave

### 1.2 Tasks

- `@ray.remote` turns a function into a **task**.
- `.remote()` schedules it on any node with a free CPU and immediately returns an **`ObjectRef`**.
- `ray.get` waits for the result.

Below: 16 steps of 1 second each, first in plain Python, then as 16 Ray tasks. The first run may include
node startup; run the cell again to time the tasks alone.

In [ ]:
def slow_step(i):
    time.sleep(1)          # stands in for 1 second of work, such as featurizing a molecule
    return i * i

t0 = time.time()
serial = [slow_step(i) for i in range(16)]
print(f"plain Python: {time.time() - t0:4.1f} s")

slow_task = ray.remote(slow_step)            # the same function as a Ray task
t0 = time.time()
refs = [slow_task.remote(i) for i in range(16)]
print(f"16 tasks submitted in {time.time() - t0:.3f} s, first reference: {refs[0]}")
parallel = ray.get(refs)
print(f"Ray tasks:    {time.time() - t0:4.1f} s, identical results: {parallel == serial}")

A realistic version: RDKit descriptors for every compound in the 3 campaigns, in 16 chunks across the
cluster. Each chunk records the node it ran on.

In [ ]:
@ray.remote(num_cpus=1)
def describe(smiles: list[str]) -> pd.DataFrame:
    # Molecular weight, ring count and hydrogen-bond donors for each SMILES string
    import socket
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Lipinski
    rows = []
    for s in smiles:
        m = Chem.MolFromSmiles(s)
        rows.append({"smiles": s, "mol_weight": Descriptors.MolWt(m) if m else None,
                     "rings": Lipinski.RingCount(m) if m else None,
                     "hbond_donors": Lipinski.NumHDonors(m) if m else None, "node": socket.gethostname()})
    return pd.DataFrame(rows)

smiles = pd.concat([pd.read_csv(TARGETS[t].path("metadata")).smiles.drop_duplicates() for t in wh.TARGET_KEYS])
chunks = np.array_split(smiles.to_numpy(), 16)

t0 = time.time()
props = pd.concat(ray.get([describe.remote(list(c)) for c in chunks]))
print(f"{len(props)} molecules, 16 tasks, {time.time() - t0:.1f} s, {props.node.nunique()} node(s)")
props.drop(columns="node").describe().round(1)

### 1.3 Resources and GPUs

- Each task or actor declares what it needs: `num_cpus`, `num_gpus`, and optionally a GPU model via `accelerator_type`.
- Ray places it where those resources are free; if none are, the autoscaler starts a node.
- GPU requests can be fractional: `num_gpus=0.5` puts 2 processes on 1 GPU. Section 3 uses this so 1 predictor
  computes while the other prepares its next molecule on the CPU.

The next cell requests 2 tasks of half an L4 each and moves on without waiting. If no L4 node is running,
the autoscaler starts a node now, so it is ready for Section 3. With `USE_GPUS = False` the request is skipped.

In [ ]:
@ray.remote(num_gpus=0.5)
def which_gpu(name):
    import socket, subprocess
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    return {"task": name, "node": socket.gethostname(), "GPU ids assigned by Ray": ray.get_gpu_ids(), "GPU": gpu}

gpu_refs = []
if USE_GPUS:
    gpu_refs = [which_gpu.options(accelerator_type=GPU).remote(n) for n in ("predictor A", "predictor B")]   # returns immediately
    print(f"Requested 2 x 0.5 {GPU}. GPUs free right now: {ray.available_resources().get('GPU', 0)}")
else:
    print("USE_GPUS is False: the GPU request runs on the presenter's cluster.")

### 1.4 Objects

Task inputs and outputs live in the object store. A large input shared by many tasks (a metadata table,
fingerprints, model weights) is stored once with `ray.put`; every task receives the same `ObjectRef` and
reads the copy on its own node.

<img src="docs/images/ray/objects.png" alt="Objects in Ray" width="300">
<sub>Diagram: Ray documentation, docs.ray.io (Apache 2.0).</sub>

In [ ]:
meta_all = pd.concat([pd.read_csv(TARGETS[t].path("metadata")).assign(target=t) for t in wh.TARGET_KEYS])
meta_ref = ray.put(meta_all)                     # stored once in the object store
print(f"metadata: {len(meta_all)} rows, stored as {meta_ref}")

@ray.remote
def count_phase(meta: pd.DataFrame, target: str) -> dict:
    # Ray passes the table itself to the task, resolved from the ObjectRef
    m = meta[meta.target == target]
    return {"target": TARGETS[target].name, "fragment-screen structures": int(m.fragment_screen.sum()),
            "follow-on structures": int((~m.fragment_screen.astype(bool)).sum())}

pd.DataFrame(ray.get([count_phase.remote(meta_ref, t) for t in wh.TARGET_KEYS]))

### 1.5 Task graphs and `ray.wait`

- An `ObjectRef` can be passed straight into another task: Ray starts it once the input is ready, so a whole
  graph of work is submitted at once.
- `ray.wait` returns results as they complete, which is useful for progress reporting.

In [ ]:
@ray.remote
def heaviest(df: pd.DataFrame) -> pd.Series:
    return df.sort_values("mol_weight").iloc[-1]

# describe, then heaviest, submitted together; no ray.get in between
chained = [heaviest.remote(describe.remote(list(c))) for c in chunks[:6]]

pending, finished = chained, 0
while pending:
    ready, pending = ray.wait(pending, num_returns=1)
    finished += 1
    top = ray.get(ready[0])
    print(f"{finished}/6 finished: heaviest molecule {top.mol_weight:6.1f} Da  {top.smiles[:50]}")

### 1.6 Actors

An **actor** is a class decorated with `@ray.remote`. Ray creates it once on 1 node; it keeps its state
between calls. Actors are the natural home for anything expensive to load, such as a co-folding model with
several GB of weights on a GPU. All predictors in Section 3 are actors.

The actor below loads the EV-A71 2A fragment screen once, then reports how similar any new compound is to
its nearest fragment.

In [ ]:
@ray.remote(num_cpus=1)
class FragmentMemory:
    def __init__(self, fragment_smiles):
        from rdkit import Chem, DataStructs
        from rdkit.Chem import rdFingerprintGenerator
        self.gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
        self.DS, self.Chem = DataStructs, Chem
        self.fps = [self.gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in fragment_smiles]   # loaded once
        self.queries = 0

    def nearest(self, smiles):
        self.queries += 1                                                                      # kept between calls
        fp = self.gen.GetFingerprint(self.Chem.MolFromSmiles(smiles))
        return max(self.DS.BulkTanimotoSimilarity(fp, self.fps))

    def stats(self):
        return f"{len(self.fps)} fragments loaded, {self.queries} queries answered"

meta_ev71 = meta_all[meta_all.target == "ev71-2a"]
memory = FragmentMemory.remote(meta_ev71[meta_ev71.fragment_screen].smiles.unique().tolist())

late = meta_ev71[~meta_ev71.fragment_screen].drop_duplicates("compound_group").tail(5)
sims = ray.get([memory.nearest.remote(s) for s in late.smiles])
print(ray.get(memory.stats.remote()))
pd.DataFrame({"compound": late.compound_group, "similarity to closest fragment (0 to 1)": np.round(sims, 2)})

Many follow-on compounds have a similarity well below 0.5 to every fragment: they contain chemistry the
fragment screen never sampled. That makes them the harder and more relevant test, and every result in this
notebook is reported separately for fragments and follow-ons.

### 1.7 Fault tolerance

On a large cluster, failures are routine: a spot instance is reclaimed, a node runs out of memory, an
external tool fails on 1 input.

- Tasks retry automatically (`max_retries`; `retry_exceptions=True` includes Python exceptions).
- Actors restart (`max_restarts`).
- Ray Train restarts a whole training run from its last checkpoint (Section 5.5).

Below, each attempt fails with 30% probability, yet all 20 tasks complete. The retry messages above the
result are expected.

In [ ]:
@ray.remote(max_retries=10, retry_exceptions=True)
def unreliable_step(i):
    import random
    if random.random() < 0.3:
        raise RuntimeError("simulated node failure")
    return i

print(f"{len(ray.get([unreliable_step.remote(i) for i in range(20)]))} of 20 tasks completed")

### 1.8 GPU placement and the 4 Ray libraries

The 2 GPU tasks from Section 1.3 have had time to start. The cell waits up to 1 minute for them.

In [ ]:
if not gpu_refs:
    print("Skipped: USE_GPUS is False.")
else:
    ready, pending = ray.wait(gpu_refs, num_returns=len(gpu_refs), timeout=60)
    if pending:
        print(f"The GPU node is still starting. Run this cell again in a few minutes. {cluster_table().GPUs.sum()} GPUs up.")
    else:
        display(pd.DataFrame(ray.get(ready)))

Each task received half an L4; `ray.get_gpu_ids()` reports which device. The rest of the workshop builds
on these primitives:

| Library | Built on | In this workshop |
|---|---|---|
| **Ray Data** | Tasks for CPU stages, actor pools for model stages, streaming through the object store | Prediction with 4 models and scoring against crystal structures |
| **Ray Train** | 1 actor per GPU in a PyTorch DDP group, with checkpoints and automatic restarts | Fine-tuning OpenFold3 on 8 GPUs |
| **Ray Serve** | Actor replicas behind an HTTP proxy, with autoscaling | The 3D results viewer and live prediction |
| **Ray Tune** | 1 trial per configuration, with early-stopping schedulers | Learning-rate and inference sweeps (Appendix) |

![Workflow](docs/images/workflow.png)

Keep the **Metrics** page of the Ray Dashboard open in a separate tab (link printed in Setup).

| Panel | Healthy | Needs attention |
|---|---|---|
| **Node GPU** | Near 100% while a model predicts | Long stretches near 0%: GPUs wait on CPU stages or model loading |
| **Node GPU Memory** | Stable, below the 24 GB of an L4 | Near the limit: lower `batch_size` or give each actor a full GPU |
| **Node Count** | GPU nodes join within minutes | No change: no capacity available |
| **Scheduler Task / Actor State** | Pending → running → finished | Many tasks pending for minutes: resources missing |
| **Object Store Memory** | Rises and falls with each batch | Grows without falling, or spills to disk |
| **Ray Data: Rows output / second** | Every operator produces rows | 1 operator at 0 while others wait: the bottleneck |

> Watch 3 panels: **GPU utilization** (are GPUs busy?), **Node Count** (have nodes started?) and
> **Scheduler Task State** (is anything stalled?).

## 2. The data: 3 drug discovery campaigns

A campaign has 2 phases:

- **Fragment screen.** Hundreds of small molecules (~15 heavy atoms) are soaked into protein crystals; each
  hit is solved as a crystal structure. Structures are cheap and plentiful here.
- **Follow-on design.** Chemists grow the best fragments into larger, tighter binders over months. Only a
  fraction is ever crystallized. **This is where a reliable predicted pose is worth the most, and where
  prediction is hardest.**

All 3 targets are viral proteases. The crystal structures are ground truth; the models see only the
protein sequence and each compound's SMILES string.

![From fragment screen to optimized compound](docs/images/campaign_explained.gif)

In [ ]:
rows, ligands = [], {}
for t in wh.TARGET_KEYS:
    meta, lig = load_ligands(TARGETS[t])
    ligands[t] = lig
    rows.append({"target": TARGETS[t].name,
                 "crystal structures": len(meta),
                 "unique compounds": len(lig),
                 **{st: int((lig.stage == st).sum()) for st in ["fragment", "early follow-on", "late follow-on"]},
                 "with measured affinity": int(lig.experimental_pKD.notna().sum())})
pd.DataFrame(rows).set_index("target")

**x7259a** is a late EV-A71 2A follow-on with high measured affinity (pKD 7.9). Its crystal structure is
below: dark sticks are the compound, thin lines the protein residues within 5 Å. Drag to rotate, scroll to zoom.

In [ ]:
ev71 = wh.load_results("ev71-2a")         # full benchmark results, one row per crystal complex
x7259a = ev71[ev71.complex_name == "A71EV2A-x7259a"].iloc[0]
wh.show_complex(x7259a, models=())

The animation follows 1 compound series from a fragment to a potent follow-on, with each model's prediction
at every step.

![A fragment grown into a potent follow-on](docs/images/campaign.gif)

## 3. Prediction and scoring at scale with Ray Data

⏱ **Budget: 20 minutes.** 3 parts:

1. **3.1–3.3** A live pipeline on 8 EV-A71 2A late follow-ons with 3 models (5–10 min).
2. **3.4** The view at scale: all 306 late follow-ons, predicted live on 12 L4 GPUs since the start of the session.
3. **3.5** What the cluster metrics revealed while we built this.

**The 3 models**

| Model | What it is |
|---|---|
| **Boltz-2** | General co-folding model; also predicts binding affinity |
| **OpenBind-0** | OpenFold3 weights released with the OpenBind data (the default checkpoint of OpenFold3 0.5) |
| **OpenFold3 145k** | General OpenFold3 model (`of3-p2-145k`) that predates the OpenBind data. **It is the starting point for the fine-tuning in Section 5** |

**How Ray Data runs it.** A `Dataset` is a table split into **blocks** in the object store; every step
processes blocks in parallel and streams them onward, so CPU and GPU steps run at the same time.

<img src="docs/images/ray/dataset-arch-with-blocks.svg" alt="Ray Data blocks" width="520">
<img src="docs/images/ray/streaming-topology.svg" alt="Streaming execution" width="520">
<sub>Diagrams: Ray documentation, docs.ray.io (Apache 2.0).</sub>

| Step | Runs on | Does |
|---|---|---|
| `validate` | CPU | Parses each SMILES with RDKit |
| `Boltz2Predictor` | ½ L4 per actor | Pose and affinity with Boltz-2 |
| `OpenBind0` | ½ L4 per actor | Pose with OpenFold3 + OpenBind-0 weights |
| `GeneralOF3` | ½ L4 per actor | Pose with OpenFold3 145k (OpenFold3 0.4.3) |
| `make_scorer` | CPU | Superposes each prediction on its crystal and computes ligand RMSD |

- Model steps are **actors** (`ActorPoolStrategy`): each loads its model once and reuses it for every batch.
- `num_gpus=0.5` puts 2 actors on each L4, so the GPU stays busy while the other actor prepares inputs.

![Pipeline architecture](docs/images/architecture.png)

<img src="docs/images/ray/data-batch-inference.png" alt="Batch inference with Ray Data" width="760">
<sub>Diagram: Ray documentation, docs.ray.io (Apache 2.0).</sub>

### 3.1 Pick the compounds

x7259a plus 7 other late follow-ons with a curated, non-covalent crystal structure.

In [ ]:
# x7259a plus seven other late follow-ons with a curated crystal structure (non-covalent)
lig = ligands["ev71-2a"]
curated = set(ev71.complex_name)
pool = lig[(lig.stage == "late follow-on") & lig.curated_any & ~lig.covalent.astype(bool) & (lig.ligand_id != "x7259a")]
picked = pd.concat([lig[lig.ligand_id == "x7259a"], pool.sample(7, random_state=1)]).reset_index(drop=True)
picked["complexes"] = picked.complexes.map(lambda cs: [c for c in cs if c in curated])
picked[["ligand_id", "smiles", "stage", "experimental_pKD"]]

### 3.2 Build the pipeline

Execution is lazy: the cell defines the pipeline; nothing runs until results are requested.

In [ ]:
MY_PRED = wh.MINE / "predictions"

class OpenBind0(OF3Predictor):
    # OpenFold3 0.5 with the OpenBind-0 weights, writing to your own folder
    def __init__(self, out_dir):
        super().__init__(key="openbind0", ckpt_path=str(OF3_OB_CKPT), target="ev71-2a", out_dir=out_dir)

class GeneralOF3(OF3Predictor):
    # OpenFold3 145k: its weights load only in OpenFold3 0.4.3, installed alongside at /opt/of3-legacy
    def __init__(self, out_dir):
        super().__init__(key="of3p2", ckpt_path=str(OF3_P2_CKPT), binary=OF3_LEGACY,
                         target="ev71-2a", out_dir=out_dir)

def gpu_step(actors):
    # `actors` model processes, each on half a GPU; each loads its model once and keeps it
    return dict(batch_size=2, batch_format="pandas", num_gpus=0.5, num_cpus=3, accelerator_type=GPU,
                compute=ray.data.ActorPoolStrategy(size=actors, max_tasks_in_flight_per_actor=1))

ds = (ray.data.from_pandas(picked[["ligand_id", "smiles", "complexes", "stage", "experimental_pKD"]])
      .repartition(4)
      .map_batches(validate, batch_format="pandas")                       # CPU
      .map_batches(Boltz2Predictor, **gpu_step(2),                         # GPU: 1 L4
                   fn_constructor_kwargs={"target": "ev71-2a", "out_dir": str(MY_PRED / "boltz2")})
      .map_batches(OpenBind0, **gpu_step(1),                               # GPU: half of a 2nd L4
                   fn_constructor_kwargs={"out_dir": str(MY_PRED / "openbind0")})
      .map_batches(GeneralOF3, **gpu_step(1),                              # GPU: the other half
                   fn_constructor_kwargs={"out_dir": str(MY_PRED / "of3p2")})
      .flat_map(make_scorer(["boltz2", "openbind0", "of3p2"], "ev71-2a")))  # CPU
ds

### 3.3 Run it

⏱ **5–10 minutes on the first run**, mostly GPU nodes starting and models loading. On the Metrics page:
*Node Count* shows nodes joining, *Node GPU* shows them predicting, *Rows output / second* shows each step's progress.

With `USE_GPUS = False` the cell loads the stored predictions for the same 8 compounds instead.

In [ ]:
RUN_LIVE = USE_GPUS

t0 = time.time()
mine = ds.to_pandas() if RUN_LIVE else ev71[ev71.ligand_id.isin(picked.ligand_id)].reset_index(drop=True)
print(f"{mine.ligand_id.nunique()} compounds, {len(mine)} crystal complexes scored in {(time.time() - t0) / 60:.1f} min")

models = ["boltz2", "openbind0", "of3p2"]
view = mine[["complex_name"] + [f"{k}_lig_rmsd" for k in models]].copy()
view.columns = ["complex"] + [f"{wh.MODEL_NAMES[k]} RMSD (Å)" for k in models]
(view.style.format(precision=2)
     .map(lambda v: "color:#2E7D4F;font-weight:600" if v < 2 else "color:#B3412E", subset=view.columns[1:]))

**Reading the table.** *Ligand RMSD* is the average distance between each predicted ligand atom and its
crystal counterpart, after superposing the predicted protein on the crystal protein.

- **< 2 Å (green): correct pose.** For scale, a C–C bond is ~1.5 Å.
- **> 2 Å (red):** usually in the pocket but turned the wrong way, which is not useful for design.

In [ ]:
# Your predictions on the crystal: dark = crystal, blue = Boltz-2, amber = OpenBind-0, slate = OpenFold3 145k
wh.show_complex(mine[mine.complex_name == "A71EV2A-x7259a"].iloc[0], models=("boltz2", "openbind0", "of3p2"))

For x7259a:

- **OpenBind-0** reproduces the crystal pose to 0.4 Å.
- **OpenFold3 145k** is in the pocket but 3.7 Å off.
- **Boltz-2** places the compound ~13 Å away, in a different region of the protein.

2 more compounds from this run:

In [ ]:
for _, row in mine.drop_duplicates("ligand_id").iloc[1:3].iterrows():
    wh.show_complex(row, models=("boltz2", "openbind0", "of3p2"), height=340)

### 3.4 At scale: all 306 late follow-ons, live

At the start of the session the presenter launched every EV-A71 2A late follow-on through Boltz-2 and
OpenBind-0 on **12 L4 GPUs** (`anyscale job submit -f jobs/workshop.yaml`).

- Each prediction is written to shared storage the moment it finishes.
- The cell below scores whatever is done so far with Ray Data on CPU nodes, and compares it with the full
  benchmark on the same compounds. Run it again to watch the count rise.

In [ ]:
# The most recent live run, or set LIVE_RUN_DIR to a specific one
LIVE_RUN_DIR = None
runs = sorted((SHARED / "workshop" / "live").glob("*"))
live_dir = Path(LIVE_RUN_DIR) if LIVE_RUN_DIR else (runs[-1] if runs else None)

late = ligands["ev71-2a"][ligands["ev71-2a"].stage == "late follow-on"].copy()
if live_dir is None:
    print("No live run found. Start one with: anyscale job submit -f jobs/workshop.yaml")
else:
    for k in ("boltz2", "openbind0"):
        cifs = [live_dir / "predictions" / k / f"{l}.cif" for l in late.ligand_id]
        late[f"{k}_cif"] = [str(c) if c.exists() else None for c in cifs]
    done = late.dropna(subset=["boltz2_cif", "openbind0_cif"]).copy()
    print(f"Live run {live_dir.name}: Boltz-2 {late.boltz2_cif.notna().sum()}/{len(late)}, "
          f"OpenBind-0 {late.openbind0_cif.notna().sum()}/{len(late)} compounds predicted")
    done["complexes"] = done.complexes.map(lambda cs: [c for c in cs if c in curated])   # curated crystals only
    done = done[done.complexes.str.len() > 0]
    if len(done):
        live = (ray.data.from_pandas(done[["ligand_id", "complexes", "boltz2_cif", "openbind0_cif"]])
                .repartition(max(1, len(done) // 8))          # eight compounds per CPU task
                .flat_map(make_scorer(["boltz2", "openbind0"], "ev71-2a")).to_pandas())   # CPU tasks
        stored = ev71[ev71.complex_name.isin(live.complex_name)]
        display(pd.DataFrame({f"live run ({len(live)} crystals)": [wh.success(live[f"{k}_lig_rmsd"]) for k in ("boltz2", "openbind0")],
                              "full benchmark, same crystals": [wh.success(stored[f"{k}_lig_rmsd"]) for k in ("boltz2", "openbind0")]},
                             index=["Boltz-2", "OpenBind-0"]).style.format("{:.1f}")
                .set_caption("Share of poses within 2 Å of the crystal structure (%)"))

### 3.5 What the cluster told us

The Metrics page is how we tuned this pipeline. 3 findings from building it:

| Observation | Cause | Fix |
|---|---|---|
| 2 OpenFold3 processes per L4 nearly **double** throughput (181 → 327 ligands/hour) | 1 process leaves the GPU idle while it featurizes on CPU | `num_gpus=0.5` |
| OpenFold3 runs **1.44×** faster on a 16-vCPU node than an 8-vCPU node with the same L4 | Featurization is CPU-bound | `g6.4xlarge` for GPU workers |
| A 699-ligand job cycled through **43 nodes** at ~40% GPU utilization | 3 OpenFold3 0.4.3 processes exhausted the 32 GB host RAM; the OS killed Ray on the node | 64 GB nodes, 2 processes per GPU → **402 new predictions in 18 min on 12 L4 at 80% utilization** |

![Throughput](docs/images/throughput.png)
![GPU cluster](docs/images/gpu_cluster.png)

## 4. Results: where general models fall short

The same pipeline ran on all 3 campaigns before the workshop (1–2 hours per target on 8 L4 GPUs). Results use
the **curated set**: a valid crystal reference and no suspected crystallographic artefact. Every number below
is measured in this pipeline, with 1 predicted pose per compound.

In [ ]:
results = {t: wh.load_results(t) for t in wh.TARGET_KEYS}
table = wh.glance(results, models=("boltz2", "openbind0", "of3p2"))
print("Share of poses within 2 Å of the crystal structure (%)")
table.style.format("{:.1f}", na_rep="—").background_gradient(cmap="Greens", vmin=0, vmax=100, axis=None)

**What the table shows**

- **Accuracy depends on the target, not just the model.** Boltz-2 and OpenFold3 145k are accurate on both
  coronavirus proteases (84–87% of follow-ons) but fail on EV-A71 2A (3% and 36%).
- **OpenBind-0 is the exception: 93% on EV-A71 2A follow-ons.** Read this number with care:
  - OpenBind-0 comes from the consortium that generated the EV-A71 2A data, and its training cutoff (June 2025)
    is later than almost all of these structures.
  - On SARS-CoV-2 it performs like the general models (89%), and on EV-A71 2A *fragments* it reaches only 28%.
  - Its EV-A71 2A accuracy most likely reflects related training data, so we treat it as an **in-distribution
    reference**, not a test of generalization.
- **The practical question for a new project:** your target is not in anyone's training set. What does it take
  to bring a general model from 24% to a useful level? Section 5 answers that.

In [ ]:
wh.by_stage(results["ev71-2a"], models=("boltz2", "openbind0", "of3p2")).style.format("{:.1f}") \
  .set_caption("EV-A71 2A: share of poses within 2 Å by campaign stage (%)")

Late follow-ons are the hardest stage: OpenFold3 145k falls from 48% on early follow-ons to **24%**, and to 13%
on the latest fifth of the campaign.

![Pose accuracy](docs/images/pose_accuracy.png)

**Binding affinity.** Boltz-2 also predicts affinity. Spearman ρ measures how well its predictions *rank*
compounds against measured values (1 = identical ranking, 0 = no relationship).

In [ ]:
rho = {}
for t, df in results.items():
    a = df.drop_duplicates("ligand_id")[["experimental_pKD", "boltz2_affinity_pred_value"]].astype(float).dropna()
    # Boltz-2 reports log10(IC50 in µM), where lower means tighter binding, so the sign is flipped
    rho[TARGETS[t].name] = {"Spearman": round((-a.boltz2_affinity_pred_value).corr(a.experimental_pKD, "spearman"), 2),
                            "compounds": len(a), "measurement": TARGETS[t].affinity_label}
pd.DataFrame(rho).T

![Affinity](docs/images/affinity.png)

**Explore.** Enter any complex from the 3 targets, for example `"A71EV2A-x7056a"`, `"MERS-a1994b"` or `"SARS-5rf4-a"`.

In [ ]:
CID = "A71EV2A-x7056a"
row = pd.concat(results.values()).query("complex_name == @CID").iloc[0]
wh.show_complex(row, models=("boltz2", "openbind0", "of3p2"))

## 5. Fine-tuning OpenFold3 on fragment structures with Ray Train

⏱ **Budget: 20 minutes.** A full fine-tune takes ~2 hours on 8 GPUs, so this section works like a cooking show:

1. **5.1–5.3** The starting point, the data and the code.
2. **5.4** A short live run on the cluster (~10 min) to see Ray Train at work.
3. **5.5** Fault tolerance: what happened when a node disappeared mid-run.
4. **5.6–5.7** Results of the full runs trained before the workshop.

### 5.1 The starting point

OpenFold3 145k is a capable general model on a target it has effectively seen before, and weak on a new target:

![Before and after fine-tuning](docs/images/before_after_accuracy.gif)

| Late follow-ons, % within 2 Å | SARS-CoV-2 Mpro | EV-A71 2A |
|---|---|---|
| OpenFold3 145k | 88% | **24%** |

### 5.2 The training data: 66 fragment structures

- **What:** the 66 curated EV-A71 2A fragment-screen complexes (protein, ligand and MSA).
- **Why fragments:** they come first in every campaign and are the cheapest structures to obtain.
- **What stays held out:** every follow-on compound. All 596 follow-ons in the evaluation are compounds the model
  never sees in training, so the result measures what a team could achieve early in a real campaign.
- **Same starting point and data as the OpenBind team's released fine-tune**, which makes the 2 directly comparable.

### 5.3 From OpenFold3 training to Ray Train

OpenFold3's own training stack (data module, losses, per-sample gradient clipping, EMA) runs **unchanged**.
Ray Train replaces the launcher (`torchrun` or SLURM) and adds the cluster:

```python
trainer = TorchTrainer(
    train_loop_per_worker,                    # OpenFold3's Lightning runner with RayDDPStrategy
    scaling_config=ScalingConfig(num_workers=8, use_gpu=True, accelerator_type="L4"),
    run_config=RunConfig(storage_path=...,
                         failure_config=FailureConfig(max_failures=2),        # survive lost nodes
                         checkpoint_config=CheckpointConfig(num_to_keep=1)))
result = trainer.fit()
```

| What Ray Train adds | Evidence from our runs |
|---|---|
| **Multi-node with 1 flag** | 2 → 8 GPUs is `--num-workers 8`. A step takes 7.6 s on 2 GPUs and ~8 s on 8 separate nodes: cross-node synchronization is nearly free |
| **Observability** | Every step reports loss, protein–ligand lDDT, step time, GPU memory (13.2 GB of 24) and GPU utilization (~81%) |
| **Fault tolerance** | Lost workers are restarted automatically; with checkpoints every 100 steps, training resumes where it stopped (5.5) |
| **Any GPU, any zone** | The same script ran on L4 and L40S, across availability zones, on spot or on-demand capacity |
| **Older software stacks** | 145k loads only in OpenFold3 0.4.3; Ray runs the workers in that environment via `runtime_env={"py_executable": ...}` |

<img src="docs/images/ray/train-overview.png" alt="Ray Train" width="560">
<sub>Diagram: Ray documentation, docs.ray.io (Apache 2.0).</sub>

### 5.4 Live: a short fine-tune on the cluster

⏱ **~10 minutes** with warm GPU nodes: `FT_STEPS` steps on `FT_GPUS` L4 GPUs, starting from OpenFold3 145k.

- Each line is 1 optimizer step, averaged over all workers.
- Set `FT_DEMO_FAILURE = True` to crash the run on purpose at step 25 and watch Ray Train resume from the
  checkpoint at step 20.
- With `USE_GPUS = False` the cell skips training; the plot below shows the full recorded run.

In [ ]:
RUN_FINETUNE = USE_GPUS
FT_GPUS, FT_STEPS = 4, 40
FT_DEMO_FAILURE = False

live_run = None
if RUN_FINETUNE:
    live_run = f"workshop-{wh.USER}-{time.strftime('%H%M%S')}"
    cmd = [sys.executable, "finetune_ray_train.py", "--base", "p2", "--accelerator", GPU,
           "--num-workers", str(FT_GPUS), "--max-steps", str(FT_STEPS), "--checkpoint-every", "20",
           "--storage", str(wh.MINE / "finetune"), "--name", live_run]
    if FT_DEMO_FAILURE:
        cmd += ["--fail-at-step", "25"]
    # finetune_ray_train.py moves itself into OpenFold3 0.4.3 for the 145k base; stream its progress here
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, "AIDDD_SHARED": str(SHARED)})
    for line in proc.stdout:
        if any(k in line for k in (" step ", "Resuming", "simulated failure", "Done in", "weights:", "run ")):
            print(line.rstrip()[-140:], flush=True)
    proc.wait()
else:
    print("USE_GPUS is False: the plot below shows the full recorded run.")

In [ ]:
import matplotlib.pyplot as plt

full = wh.loss_curve(wh.ours_run())
yours = None
if live_run:
    found = sorted((wh.MINE / "finetune" / live_run).glob("checkpoint_*/loss_curve.json"))
    yours = pd.DataFrame(json.loads(found[-1].read_text())) if found else None

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, col, name in zip(axes, ["loss", "lddt_inter_protein_ligand_step"],
                         ["Training loss (lower is better)", "Protein–ligand lDDT (higher is better)"]):
    if full is not None:
        r = full[full.step > 1]
        ax.plot(r.step, r[col], color=wh.COLORS["of3ray"], alpha=0.2, lw=0.8)
        ax.plot(r.step, r[col].rolling(20, min_periods=5).mean(), color=wh.COLORS["of3ray"], lw=2,
                label=f"full run, {int(r.step.max())} steps")
    if yours is not None:
        y = yours[yours.step > 1]
        ax.plot(y.step, y[col], "o-", color="#B3412E", ms=3, lw=1.2, label=f"your run, {int(y.step.max())} steps")
    ax.set_title(name, loc="left", fontsize=11)
    ax.set_xlabel("optimizer step")
    ax.grid(alpha=0.3)
axes[0].legend(frameon=False)
plt.tight_layout()

### 5.5 Fault tolerance: the night a node disappeared

While preparing this workshop, a spot instance under 1 of the 8 GPUs was reclaimed at step ~700 of a
1,000-step run. Ray Train detected it and restarted all workers automatically:

```
Training was interrupted by node preemption (preempted_ranks=[2])
[State Transition] PREEMPTING -> RESTARTING
```

- **Before:** weights were saved only at the end, so the restart began again at step 0, losing ~1.5 hours.
- **After:** the script reports a checkpoint every 100 steps. A restarted worker calls `ray.train.get_checkpoint()`,
  loads the latest weights, resumes the learning-rate schedule and trains only the remaining steps. At most
  100 steps (~13 minutes) are repeated; each checkpoint costs ~8 seconds.

![Restart from 0 vs resume from a checkpoint](docs/images/fault_tolerance.gif)

**1. A worker node fails mid-training.**

<img src="docs/images/ray/train-worker-node-failure.png" alt="A worker node fails" width="860">

**2. Ray Train restarts the worker group and resumes from the latest checkpoint.**

<img src="docs/images/ray/train-worker-group-recovery.png" alt="The worker group recovers from the checkpoint" width="860">

<sub>Diagrams: Ray documentation, docs.ray.io (Apache 2.0).</sub>

```python
resume = ray.train.get_checkpoint()            # None on a fresh start
if resume:
    start_step = json.loads((Path(resume.to_directory()) / "resume.json").read_text())["step"]
...
ray.train.report(metrics, checkpoint=ray.train.Checkpoint.from_directory(tmp))   # every 100 steps
```

### 5.6 Results: before and after fine-tuning

The full runs, trained before the workshop and scored on every follow-on compound:

| Run | Data | GPUs × steps | Diffusion samples / step |
|---|---|---|---|
| **Ours (Ray Train)** | 66 EV-A71 2A fragments | 8 L4 × 1,000 | 8 |
| **Ours, paper recipe (Ray Train)** | 80 EV-A71 2A fragments | 8 L40S × 1,000 | 32 |
| **OpenBind team (released)** | ~79 EV-A71 2A fragments | 8 GPUs × 1,000 | 32 |

All 3 start from OpenFold3 145k.

In [ ]:
print(f"Our fine-tune: {wh.ours_run() or 'not benchmarked yet'}")
wh.before_after(results).style.format(lambda v: v if isinstance(v, str) else f"{v:.1f}") \
  .set_caption("Late follow-ons: share of poses within 2 Å (%)")

**How to read it**

- **EV-A71 2A column:** the effect of fine-tuning on held-out compounds of the target it was trained for.
- **SARS-CoV-2 and MERS-CoV columns:** a check on the targets it was *not* trained for. The released fine-tune
  drops there (88% → 57% on SARS-CoV-2): a fine-tuned model is a specialist for its own target.
- Rows marked *pending* fill in automatically once that run has been scored.

The same late follow-on compound before and after fine-tuning:

![The same compound before and after fine-tuning](docs/images/before_after_pose.gif)

In [ ]:
# x7056a, a late follow-on: slate = 145k, teal = our fine-tune, red = OpenBind release, dark = crystal
row = results["ev71-2a"].query("complex_name == 'A71EV2A-x7056a'").iloc[0]
wh.show_complex(row, models=("of3p2", "of3ray", "of3ft"))

### 5.7 What it cost

| | Our main run | Paper recipe |
|---|---|---|
| GPUs | 8 L4 | 8 L40S |
| Steps | 1,000 | 1,000 |
| Time per step | ~8 s | ~5.5 s |
| Wall time | ~2.3 h | ~1.6 h |
| GPU hours | ~18 | ~13 |

To fine-tune and benchmark your own checkpoint as Anyscale Jobs:

```bash
anyscale job submit -f jobs/job.yaml --name ft-p2-evfrag -- \
  python finetune_ray_train.py --base p2 --num-workers 8 --max-steps 1000 --checkpoint-every 100
python scripts/bench_checkpoints.py --runs <run name> --targets ev71-2a,sars2-mpro
```

## 6. Serving the fine-tuned model with Ray Serve

⏱ **Budget: 17 minutes.** 2 parts:

1. **6.1 to 6.3** The results viewer and live prediction.
2. **6.4** Swapping checkpoints and rolling out a new model safely.

`serve_viewer.py` is 1 Ray Serve application built from several deployments, each with its own resources
and scaling:

| Deployment | Runs on | Does |
|---|---|---|
| `Viewer` | 1 CPU | Web page: every crystal complex, a 3D view of each prediction, a SMILES box |
| Pose model | ½ L4 per replica | Predicts the pose of a new molecule with OpenFold3 |
| `Boltz2Live` | ½ L4 per replica | Predicts binding affinity with Boltz-2 |

**What to notice**

- **Pre- and post-processing are ordinary Python.** SMILES validation runs before the GPU models; pose
  rendering and the plain-language explanation run after. Each lives in the deployment that needs it.
- **Composition.** `Viewer` holds a handle to each model and calls both at once, so a request takes as long as
  the slower model, not the sum.
- **GPUs from every zone.** Model replicas scale between 1 and 4 and start in any availability zone with
  capacity (`enable_cross_zone_scaling` in `jobs/service.yaml`).
- **Same code, 2 settings:** `serve.run` in this Workspace, an Anyscale Service in production.

<img src="docs/images/ray/serve-model-composition.png" alt="Composing pre-processing, models and post-processing" width="1000">
<img src="docs/images/ray/serve-autoscaling.svg" alt="Ray Serve autoscaling" width="480">
<sub>Diagrams: Ray documentation, docs.ray.io (Apache 2.0).</sub>

### 6.1 Open the viewer

⏱ **1–3 minutes** if a GPU node needs to start. With `USE_GPUS = False` the cell connects to the shared service.

In [ ]:
from ray import serve
import serve_viewer

if USE_GPUS:
    serve.run(serve_viewer.build_app(accelerator=GPU), name="aiddd", route_prefix="/", blocking=False)
    viewer_url, api_url = wh.serve_link(), "http://localhost:8000"
else:
    assert SERVICE_URL, "Set SERVICE_URL in Setup to the address the presenter shares, and run Setup again."
    viewer_url = api_url = SERVICE_URL.rstrip("/")
    print(f"Shared service: {viewer_url}")

In the viewer, choose a model tab and click a complex: the predicted compound (green) appears on the crystal
structure (orange).

### 6.2 Molecules to try

Paste any of these into the viewer's SMILES box, or into `MY_SMILES` below. All are EV-A71 2A late
follow-ons with a crystal structure, so you can compare the prediction with the experiment.

| Compound | Why it is interesting | SMILES |
|---|---|---|
| **x7056a** | 145k misses by 6.0 Å; fine-tuned models land within 0.4 Å | `CSc1ncnc(N2CCC3(CCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x7101a** | Same series with a larger ring; 145k misses by 6.4 Å | `CSc1ncnc(N2CCC3(CCCCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x7410a** | Potent (pKD 6.8), a different series; 145k misses by 5.6 Å | `CSc1ccc(-c2cnn[nH]2)cc1OC(=O)N1C[C@@H](C(N)=O)C[C@@H]2C[C@@H]21` |
| **x6714a** | Smaller scaffold; 145k misses by 4.6 Å | `NC(=O)[C@H]1CCC(=O)N(c2cc3cnccc3c(F)c2F)C1` |
| **x7259a** | Most potent shown here (pKD 7.9); Boltz-2 misses by 13 Å | `CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x7259a, S → O** | A new molecule with no crystal structure | `COc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |

- A new molecule takes 1–3 minutes; a repeated molecule returns instantly from the cache.
- On the shared service the whole room shares the same GPUs, so requests may queue.

### 6.3 Predict from Python

`predict` sends 1 SMILES string to the service. The optional `model` argument selects a checkpoint when the
service offers more than 1 (Section 6.4); without it the service uses its default model.

In [ ]:
import requests

def predict(smiles, model=None):
    t0 = time.time()
    headers = {"serve_multiplexed_model_id": model} if model else {}
    body = {"smiles": smiles, **({"model": model} if model else {})}
    r = requests.post(f"{api_url}/predict", json=body, headers=headers, timeout=900)
    out = r.json()
    if r.status_code != 200:
        raise RuntimeError(out.get("error", out))
    m = out.get("metrics", {})
    print(f"{out.get('status', 'done')} in {time.time() - t0:.0f} s · pose: {out.get('pose_model', model or 'default')}"
          f" · ipTM {out.get('pose_iptm') or float('nan'):.2f}"
          f" · P(binder) {m.get('affinity_probability_binary', float('nan')):.2f} (Boltz-2)")
    if out.get("cif"):
        out["cif_path"] = wh.MINE / "live" / f"{out.get('ligand_id', 'pred')}-{model or 'default'}.cif"
        out["cif_path"].parent.mkdir(parents=True, exist_ok=True)
        out["cif_path"].write_text(out["cif"])
    return out

NEW = "COc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1"     # x7259a with S replaced by O
out = predict(NEW)
if out.get("cif_path"):
    wh.show_analog(out["cif_path"], x7259a, label=f"new molecule ({out.get('pose_model', 'default')}, live)")

Grey ribbon and dark sticks: the crystal structure of x7259a. Green: the prediction for the new molecule,
superposed on that crystal. If the new molecule keeps its parent's binding mode, the 2 overlap.

**Your turn.** Replace `MY_SMILES` with any molecule from the table above, or your own (up to 300 characters).

In [ ]:
MY_SMILES = "CSc1ncnc(N2CCC3(CCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1"     # x7056a
out = predict(MY_SMILES)
if out.get("cif"):
    wh.show_prediction(out["cif"], label=f"{MY_SMILES[:40]}… ({out.get('pose_model', 'default')})")

### 6.4 Swapping checkpoints and rolling out a new model

The fine-tuned model is ready. How does it reach chemists without breaking what they use today?

**Swap checkpoints on 1 deployment.** Ray Serve **model multiplexing** serves several checkpoints from the same
replicas and loads each on demand. A request names its checkpoint; the replica keeps the most recent ones in memory.

```python
@serve.deployment(ray_actor_options={"num_gpus": 0.5})
class PoseModel:
    @serve.multiplexed(max_num_models_per_replica=3)
    async def get_model(self, model_id: str):
        return load_checkpoint(CHECKPOINTS[model_id])     # 145k, ours, OpenBind release

    async def __call__(self, smiles: str):
        model = await self.get_model(serve.get_multiplexed_model_id())
        return model.predict(smiles)
```

**The demo:** the same compound (x7056a), 3 checkpoints. The pose moves from 6.0 Å off to within 0.5 Å.

In [ ]:
# Checkpoint names offered by the service; the presenter confirms these on the day
CHECKPOINTS = ["of3p2-145k", "ours", "of3p2-ft-ev2a"]
SWAP_DEMO = False     # True once the service offers multiple checkpoints

if SWAP_DEMO:
    for ck in CHECKPOINTS:
        out = predict(MY_SMILES, model=ck)
        if out.get("cif_path"):
            wh.show_analog(out["cif_path"], results["ev71-2a"].query("complex_name == 'A71EV2A-x7056a'").iloc[0],
                           label=f"x7056a predicted by {ck}", height=340)
else:
    print("Set SWAP_DEMO = True when the service offers several checkpoints.")

**Roll out safely with a canary.** An Anyscale Service can run the new version next to the current version and send
it a share of traffic first (for example 10%). Watch latency and errors, then promote it to 100% or roll back.

```bash
anyscale service deploy -f jobs/service.yaml --canary-percent 10     # new version takes 10% of traffic
anyscale service deploy -f jobs/service.yaml --canary-percent 100    # promote
anyscale service rollback -n aiddd-viewer                       # or roll back
```

| What Ray Serve and Anyscale add | Why it matters |
|---|---|
| **GPUs from every zone** | When 1 zone runs out of L4s, replicas start in another |
| **Composition in Python** | Pre-processing, models and post-processing each scale on their own |
| **Model multiplexing** | Several checkpoints on the same GPUs, chosen per request |
| **Canary rollouts and rollback** | A new model reaches a share of users first |

## 7. Summary and next steps

- **Generalization:** general models that do well on SARS-CoV-2 drop sharply on EV-A71 2A.
- **Fine-tuning:** 66 fragment structures are enough to move OpenFold3 on held-out follow-ons (Section 5.6).
- **Ray Data:** the full benchmark, and 402 new OpenFold3 predictions in 18 minutes on 12 L4 GPUs.
- **Ray Train:** OpenFold3's own training code on 8 GPUs, with checkpoints and automatic restarts.
- **Ray Serve:** the model behind 1 endpoint, with checkpoint swaps and canary rollouts.

The [README](README.md) has the scientific background and the complete runbook.

Run the next cell to stop the service and release the GPU nodes.

In [ ]:
if USE_GPUS:
    serve.shutdown()
    print("Serve application stopped. Idle GPU nodes will be released within a few minutes.")
else:
    print("Nothing to stop: this notebook used the shared service.")

## Appendix A. Hyperparameter and inference sweeps with Ray Tune

Ray Tune runs many trials in parallel and stops the ones that fall behind. Both sweeps ran before the
workshop: `scripts/tune_finetune.py` (35 minutes on 4 L4 GPUs) and `scripts/tune_boltz.py` (62 minutes on 8 L4 GPUs).

<img src="docs/images/ray/tune_flow.png" alt="Ray Tune" width="620">
<sub>Diagram: Ray documentation, docs.ray.io (Apache 2.0).</sub>

**Sweep 1: learning rate.** Each trial is a full Ray Train run starting from OpenBind-0. The ASHA scheduler stops
trials whose lDDT falls behind; the survivor is scored on 32 held-out early follow-on complexes.

In [ ]:
lr = pd.read_csv(SHARED / "finetune" / "runs" / "tune-of3-ev71" / "summary.csv")
lr["result"] = np.where(lr.val_success.notna(),
                        (100 * lr.val_success).round(1).astype(str) + "% within 2 Å on held-out compounds",
                        "stopped by ASHA at step " + lr.steps.astype(int).astype(str))
lr[["lr", "steps", "lddt_avg20", "result"]].style.format({"lr": "{:.0e}", "lddt_avg20": "{:.3f}"}).hide()

ASHA stopped 3 of 4 trials early, so the sweep used about 450 GPU-steps instead of 600.

**Sweep 2: Boltz-2 inference settings on EV-A71 2A.** Each trial is 1 combination of recycling steps, diffusion
samples and steering; all 8 ran at once on 8 L4 GPUs.

In [ ]:
bz = pd.read_csv(SHARED / "tune" / "runs" / "tune-boltz2-ev71" / "summary.csv")
bz["% within 2 Å"] = (100 * bz.success).round(1)
bz["GPU minutes"] = bz.minutes.round(0)
(bz[["setting", "% within 2 Å", "median_rmsd", "GPU minutes"]].rename(columns={"median_rmsd": "median RMSD (Å)"})
   .style.format({"% within 2 Å": "{:.1f}", "median RMSD (Å)": "{:.1f}", "GPU minutes": "{:.0f}"}).bar(subset=["% within 2 Å"], color="#9fb6cf", vmin=0, vmax=100).hide())

The best setting lifts Boltz-2 from 0% to 13.5% at about 6 times the GPU cost. On this target, the choice of
model and fine-tuning matter far more than extra inference compute.